**STEP 1: Simulation**

In [1]:
import numpy as np
import random
from scipy.stats import wishart
def simulate_matrix(n, d, no_components,rho,abs_bound_X):
    ###Form the X matrix
    X = np.zeros((0, d))
    # Use base_value and reminder to evenly distributed the number of samples in each component
    def generate_components(n, no_components):
        base_value = n // no_components
        remainder = n % no_components
        components = [base_value] * no_components
        for i in range(remainder):
            components[i] += 1
        random.shuffle(components)
        if sum(components) > n:
            components[n-1] -= sum(components) - n
        elif sum(components) < n:
            components[n-1] += n - sum(components)
        return components
    # Generate these multivariate gaussian components in different levels of mean and cov.
    num_list = generate_components(n, no_components)
    for i in range(no_components):
        # each level's mean value, grow by 4's power.
        mean_value = 5**(i+1)
        # 0.8 probability of 1 and 0.2 probability of 0 for creating a sparse mean vector.
        mean_vector = np.random.binomial(1, 0.8, size = d)
        mean = mean_value * mean_vector
        # randomly generate a cov matrix and make sure that it is symmetric.
        cov = wishart.rvs(df=d, scale=np.eye(d))
        # add a small value to the diagonal to ensure positive-definiteness
        cov += np.eye(d) * 1e-6
        # generate a component of X.
        X_ = np.random.multivariate_normal(mean, cov, num_list[i])
        X_ = np.array(X_)
        # append the component to the list.
        X = np.vstack((X, X_))
    X = np.array(X)
    # Set the elements < 3 to 0 to make it a sparse matrix
    X[abs(X) < abs_bound_X] = 0
    ###Form the fixed beta
    np.random.seed(39)
    beta_value = np.random.binomial(1, 0.8, size = d)
    beta = (beta_value * np.random.uniform(-1, 1, size = d)).reshape(-1,1)
    beta = np.array(beta)
    np.random.seed(None)
    ###Form the error term - epsilon, which has the auto-correlation structure in time series analysis.
    r = rho # autocorrelation coefficient
    epsilon_matrix = np.zeros((0,1)) # define a zero initial matrix for vstack
    ###Form the error term in the same levels
    for i in range(no_components):
        epsilon = np.zeros((num_list[i],1))
        epsilon[0] = np.random.normal((5**(i+1))/100,1)
        for t in range(1,num_list[i]):
            epsilon[t] = r * epsilon[t-1] + np.random.normal((5**(i+1))/100,1)
        epsilon = np.array(epsilon)
        epsilon_matrix = np.vstack((epsilon_matrix, epsilon))
    epsilon_matrix = np.array(epsilon_matrix)
    epsilon = epsilon_matrix
    ###Form the Y matrix
    Y = np.array(X @ beta + epsilon)
    ###Count the number of elements < 0.1 in X and Y
    return X,Y,beta,epsilon

# Matrix Settled
n = 10**4       # number of rows         
d = 500        # number of columns
no_components = 4 # number of levels of multivariate gaussian distribution in X
rho = 0.7 # autocorrelation coefficient in time series analysis
abs_bound_X = 2 # absolute bound for X to set elements to be 0
X,Y,beta,epsilon = simulate_matrix(n, d, no_components,rho,abs_bound_X)
print("The simulated matrix X is:")
print(X)
print("The simulated vector Y is:")
print(Y)

The simulated matrix X is:
[[-34.9100235   23.11075385   2.75485143 ...  37.26622629 -46.30719532
   34.9705507 ]
 [  7.53259577 -15.7291017   36.85139089 ...   0.          -2.46111631
   23.4722465 ]
 [ -8.47398143  26.94322845  34.6355284  ...  34.13357588   0.
    0.        ]
 ...
 [617.84948088 626.01630246 637.15671525 ... 626.00027675 607.31545973
  596.52032754]
 [631.96240186 628.7309161  607.41596533 ... 586.33868937 650.40360226
  594.34982144]
 [598.10350682 613.6693104  639.69475851 ... 634.77567262 590.97230065
  641.84136277]]
The simulated vector Y is:
[[ 66.78945375]
 [  9.15495068]
 [134.57430067]
 ...
 [304.98725083]
 [455.99708064]
 [296.57573125]]


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.linalg import cho_factor, cho_solve
import time
def solver(A,b,solver):
    # cholesky decomposition algorithm
    if solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b
    return x_star
start_cholesky = time.time()
x_star_cholesky = solver(X,Y,'cholesky')
end_cholesky = time.time()
execution_cholesky = end_cholesky - start_cholesky
start_direct = time.time()
x_star_direct = solver(X,Y,'direct')
end_direct = time.time()
execution_direct = end_direct - start_direct
print("The execution time of cholesky decomposition is:")
print(execution_cholesky)
print("The execution time of direct solver is:")
print(execution_direct)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by cholesky and direct solver respectively are:")
print(norm_cholesky, norm_direct)
minimum_norm_square = min(norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)
print("Their difference by cholesky minus direct solver is:")
print(norm_cholesky - norm_direct)

The execution time of cholesky decomposition is:
0.014668464660644531
The execution time of direct solver is:
0.16509175300598145
The norm square of x_star calculated as by cholesky and direct solver respectively are:
18822.912873043646 18822.91287304345
Here we output the minimum of the norm suqare for x_star calculated as:
18822.91287304345
Their difference by cholesky minus direct solver is:
1.964508555829525e-10


###It is obvious that norm difference of cholsky and direct solver is very small.

**Consider the number of rows (n) of the matrix X and Y in the scale of powers of 10.**<br>
*In my computation environment, when n = 10^6, the computation time of simulation is only 15.2s in one test case.*<br>
*X_star by cholesky takes 0.939s and x_star by direct solver takes 2.464s*<br>
*But when n = 10^7, the computation time of simulation is as large as 21m 9.5s in one test case.*<br>
*X_star by cholesky and x_star by direct solver even collapse in time and memory.*

In [4]:
#Simulation time when n = 10^6
from IPython.display import Markdown
Markdown('![Image Description](./10^6_simulation.png)')

![Image Description](./10^6_simulation.png)

In [5]:
#Cholesky vs direct solver time when n = 10^6
Markdown('![Image Description](./10^6_two times.png)')

![Image Description](./10^6_two times.png)

In [6]:
#Simulation time when n = 10^7
Markdown('![Image Description](./10^7_simulation.png)')

![Image Description](./10^7_simulation.png)

In [7]:
###From the above different norm and optimal solver calculations, it is obvious that:
###the direct solver and cholesky decomposition are the optimal solvers.
###Since their norms are very very very nearly the same
###we choose the direct solver as our optimal solver when matrix is not so large.
###And we choose the cholesky decomposition as our optimal solver when matrix is so large.
x_Star = x_star_direct
norm_Star = norm_direct
print("The optimal solver is:")
print(x_Star)

The optimal solver is:
[[-1.16752400e-01]
 [ 7.45893548e-01]
 [ 1.20509959e-03]
 [-4.41740309e-01]
 [ 9.56545095e-01]
 [ 3.74778760e-01]
 [-1.35780082e-01]
 [-9.00980446e-01]
 [ 9.56312747e-02]
 [ 3.28447130e-04]
 [ 5.77536697e-04]
 [ 1.28052192e-03]
 [ 6.96745327e-04]
 [-1.69050212e-01]
 [ 1.16744965e-03]
 [-1.29837256e-05]
 [-6.00417743e-01]
 [ 8.65976360e-01]
 [-4.57614526e-01]
 [ 1.58680499e-03]
 [ 4.11304142e-01]
 [-1.49872773e-03]
 [-4.85201812e-01]
 [-4.06338017e-01]
 [ 8.86793790e-02]
 [ 8.55980898e-04]
 [-1.71077436e-01]
 [ 1.05980177e-01]
 [-2.57791252e-01]
 [-3.18458352e-01]
 [-1.15982789e-01]
 [ 1.37889597e-01]
 [-9.57036153e-02]
 [-1.32082816e-03]
 [ 5.99210523e-01]
 [ 4.95742196e-01]
 [ 9.62985001e-01]
 [-3.75533277e-04]
 [-6.09146533e-01]
 [-5.10754649e-04]
 [-6.21960555e-01]
 [ 3.96942980e-01]
 [ 2.07480652e-01]
 [-7.99366165e-05]
 [-6.28261023e-01]
 [-1.14962243e-01]
 [ 2.15804202e-01]
 [-6.75857467e-01]
 [-2.01370688e-01]
 [-9.24497265e-01]
 [ 8.35848583e-01]
 [ 8.838

**STEP 2: Uniform Sampling sketch // Algorithm 1: Distributed Randomized Regression**

In [8]:
# e.g. our desired sketching size is m = 1000
from operator import index


m = 1000

# Algorithm 1 inserting inside uniform sampling: Distributed Randomized Regression

# S_k @ A here is just computed as A [uniformly_sampled_index]
# As S_k here is just a diagnoal matrix of 1 or 0 where sampled rows have 1 as value
def uniform_sampling(A,b,n,m,q):
    x_hat_list = []
    for i in range(q):
        index = np.random.choice(n, size=m, replace=False)
        A_sk = A[index]
        b_sk = b[index]
        x_hat = np.linalg.inv(A_sk.T @ A_sk) @ A_sk.T @ b_sk
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar
